# 1.4 — Assignment Statements and Input 🧋

### AP CSA · Unit 1: Using Objects and Methods
**Boba Cafe Series · Lesson 4**

---

> **Setup note:** every code cell runs on the **IJava kernel** (Java 17+). Check that the kernel picker says *Java*. Each cell declares a class and then calls it with `ClassName.main(null);`.

> **Important note about `Scanner` in this notebook:** a normal Java program reads the keyboard with `new Scanner(System.in)`. That does **not** work inside a Jupyter notebook — there's no keyboard attached to a code cell, so the program would hang or crash.
>
> The fix is genuinely useful rather than a workaround: a `Scanner` can read from **any** source, including a plain `String`. Every `Scanner` cell below reads from a String that stands in for what a customer typed. **The method calls are identical either way** — only the constructor changes. Part 6 shows you the real keyboard version to use outside the notebook.

## The cafe finally gets an order screen

Every program you've written so far has had the customer's order **typed into the source code**. Want to serve a different drink? Edit the program and recompile. That's not a cafe, that's a vending machine with one button.

Today the cafe installs a real order screen. Two skills make it work:

1. **Assignment statements** — moving values around while the program runs, so a variable can start at one value and end at another.
2. **Input** — letting a human outside the program put a value *into* it.

### What you'll be able to do by the end

| # | Objective | CED reference |
|---|---|---|
| 1 | Write assignment statements with expressions and determine the value stored | 1.4.A |
| 2 | Trace a variable's value through a sequence of assignments | 1.4.A |
| 3 | Use the **accumulator** pattern to build a running total | 1.4.A |
| 4 | Swap two variables correctly | 1.4.A |
| 5 | Read input using `Scanner` methods | 1.4.B |
| 6 | Avoid the `nextInt()`-then-`nextLine()` trap | 1.4.B |

---

## Part 1 — The assignment statement, precisely

Every assignment statement in Java has the same shape:

```
variable = expression ;
```

And it always happens in this order:

> **1. Evaluate the entire right side down to a single value. 2. Store that value in the variable on the left.**

Nothing on the right side is stored until it has been reduced to one value. That ordering explains everything else in this lesson.

Read the `=` as a **left-pointing arrow**, never as "equals":

```
        total  =  price * quantity + fee
          ^                  |
          |__________________|
             the single value produced on the right
             gets stored into the variable on the left
```

In [ ]:
public class AssignmentBasics {
    public static void main(String[] args) {
        double price    = 5.75;
        int    quantity = 3;
        double fee      = 2.00;

        // The whole right side is computed first: 5.75 * 3 + 2.00 = 19.25
        // Only THEN is 19.25 stored into total.
        double total = price * quantity + fee;

        System.out.println("price    = " + price);
        System.out.println("quantity = " + quantity);
        System.out.println("fee      = $" + fee);
        System.out.println("total    = $" + total);
    }
}

AssignmentBasics.main(null);

Changing `price` **after** this line would not update `total`. The assignment already happened; `total` holds a value, not a live formula. A spreadsheet recalculates when you change a cell — Java does not.

---

## Part 2 — Tracing values (the exam skill)

"What is the value of `x` after the following code segment?" is an AP staple. The method is mechanical:

> **Keep a table. One row per statement. One column per variable. Update only the variable being assigned.**

Trace this by hand before running it:

```java
int cups = 4;
int trays = 2;
cups = cups + trays;
trays = cups * 2;
cups = trays - cups;
```

<details>
<summary><b>Worked trace (try it first)</b></summary>

| Statement | Right side evaluates to | `cups` | `trays` |
|---|---|---|---|
| `int cups = 4;` | `4` | **4** | — |
| `int trays = 2;` | `2` | 4 | **2** |
| `cups = cups + trays;` | `4 + 2` = `6` | **6** | 2 |
| `trays = cups * 2;` | `6 * 2` = `12` | 6 | **12** |
| `cups = trays - cups;` | `12 - 6` = `6` | **6** | 12 |

Final: `cups` is `6`, `trays` is `12`.

The trap in row 4: it uses the **new** value of `cups` (6), not the original 4. Always use the most recent value in your table.
</details>

In [ ]:
public class TraceValues {
    public static void main(String[] args) {
        int cups  = 4;
        int trays = 2;

        cups  = cups + trays;
        trays = cups * 2;
        cups  = trays - cups;

        System.out.println("cups  = " + cups);
        System.out.println("trays = " + trays);
    }
}

TraceValues.main(null);

---

## Part 3 — The accumulator pattern

This is the single most useful assignment pattern in the entire course:

```java
total = total + something;
```

A variable that keeps adding to itself is called an **accumulator**. It works because of the evaluation order from Part 1: the *old* value of `total` is read on the right, the addition happens, and only then does the *new* value get stored.

At the cafe, this is the running total on the register as each item is rung up.

In [ ]:
public class RunningTotal {
    public static void main(String[] args) {
        double total = 0.0;                      // accumulators start at a neutral value
        System.out.println("Starting total: $" + total);

        total = total + 5.75;                    // 0.0 + 5.75
        System.out.println("+ Milk Tea:     $" + total);

        total = total + 6.50;                    // 5.75 + 6.50
        System.out.println("+ Matcha:       $" + total);

        total = total + 0.75;                    // 12.25 + 0.75
        System.out.println("+ Extra pearls: $" + total);

        System.out.println("Order total:    $" + total);
    }
}

RunningTotal.main(null);

Two details worth locking in:

- An accumulator that sums things starts at **`0`**. One that multiplies would start at `1`. Starting a sum accumulator at anything else quietly poisons every result.
- `total = total + 0.75` is not circular reasoning. The right side is fully evaluated using the current value *before* anything is stored.

> Java has a shortcut for this: `total += 0.75`. That's Topic **1.6 — Compound Assignment Operators**, so the long form is used here on purpose.

You'll use accumulators constantly once loops arrive in Unit 2 — summing an order, counting drinks, tracking a maximum.

---

## Part 4 — Swapping two variables

Two cups got mislabeled. `cupA` says Taro, `cupB` says Matcha, and they need to trade. The obvious approach fails, and *why* it fails is the best test of whether you really understand assignment.

**Run this and watch both cups end up with the same drink.**

In [ ]:
public class BrokenSwap {
    public static void main(String[] args) {
        String cupA = "Taro";
        String cupB = "Matcha";

        System.out.println("Before: cupA = " + cupA + ", cupB = " + cupB);

        cupA = cupB;      // cupA is now "Matcha" -- "Taro" is GONE
        cupB = cupA;      // cupB gets "Matcha" back from cupA

        System.out.println("After:  cupA = " + cupA + ", cupB = " + cupB);
    }
}

BrokenSwap.main(null);

Line 1 **overwrote** `cupA`. The value `"Taro"` wasn't moved aside — it was destroyed. By line 2 there is no copy of it left anywhere in the program, so `cupB` gets `"Matcha"` handed back to it.

Think of it physically: you can't pour cup A into cup B and cup B into cup A at the same time. You need a **third cup**.

In [ ]:
public class FixedSwap {
    public static void main(String[] args) {
        String cupA = "Taro";
        String cupB = "Matcha";

        System.out.println("Before: cupA = " + cupA + ", cupB = " + cupB);

        String temp = cupA;    // park "Taro" somewhere safe
        cupA = cupB;           // cupA becomes "Matcha"
        cupB = temp;           // cupB becomes the parked "Taro"

        System.out.println("After:  cupA = " + cupA + ", cupB = " + cupB);
    }
}

FixedSwap.main(null);

The three-line temp swap works for **any** type — `int`, `double`, `String`, objects. Memorize the shape:

```java
temp = a;
a    = b;
b    = temp;
```

---

## Part 5 — What fits in what

The value produced on the right must be **compatible** with the variable's declared type on the left.

| Assignment | Legal? | Why |
|---|---|---|
| `double d = 7;` | Yes | An `int` fits inside a `double` with nothing lost — `d` becomes `7.0` |
| `int i = 7.0;` | **No** | A `double` might carry a fraction, so this loses information |

Going `int` → `double` is automatic because it's always safe. Going `double` → `int` is refused, because Java won't throw away part of your number without being explicitly told to.

In [ ]:
public class TypeFit {
    public static void main(String[] args) {
        int scoops = 3;

        double scoopsAsDouble = scoops;      // int into double: automatic and safe
        System.out.println("int 3 stored in a double: " + scoopsAsDouble);

        double price = 5.75;
        // int rounded = price;              // uncomment this line to see the compiler refuse
        System.out.println("Uncomment the line above to watch it fail.");
    }
}

TypeFit.main(null);

> Forcing the unsafe direction — writing `int rounded = (int) price;` — is **casting**, Topic **1.5**, your next lesson.

---

## Part 6 — Reading input with `Scanner`

`Scanner` is a class in the Java library that reads values from an input source. Using it takes three steps:

**Step 1 — import it** (above the class declaration):

```java
import java.util.Scanner;
```

**Step 2 — create one**, pointed at a source:

```java
Scanner scan = new Scanner(System.in);    // read from the keyboard
```

**Step 3 — call a method** for the type you want:

| Method | Returns | Reads |
|---|---|---|
| `nextInt()` | `int` | The next whole number |
| `nextDouble()` | `double` | The next decimal number |
| `nextBoolean()` | `boolean` | The next `true` or `false` |
| `next()` | `String` | The next **token** (one word, stops at whitespace) |
| `nextLine()` | `String` | The **rest of the current line** |
| `hasNext()` | `boolean` | `true` if anything is left to read |
| `close()` | — | Releases the input source when finished |

These are the methods on the official AP Java Quick Reference, so they're the ones worth knowing cold.

### The notebook version

`System.in` means "the keyboard," which a notebook cell doesn't have. But a `Scanner` will read from a `String` just as happily:

```java
Scanner scan = new Scanner("Riley 3 5.75 true");
```

Every method call after that is **exactly the same**. This isn't a toy — reading from a different source with the same methods is precisely how file reading works in Topic 4.6, where the constructor takes a `File` instead.

In [ ]:
import java.util.Scanner;

public class ScannerBasics {
    public static void main(String[] args) {
        // Stand-in for what a customer typed at the order screen
        String typedByCustomer = "Riley 3 5.75 true";

        Scanner scan = new Scanner(typedByCustomer);

        String  name     = scan.next();          // "Riley"
        int     quantity = scan.nextInt();       // 3
        double  price    = scan.nextDouble();    // 5.75
        boolean wantsIce = scan.nextBoolean();   // true

        System.out.println("Name:     " + name);
        System.out.println("Quantity: " + quantity);
        System.out.println("Price:    $" + price);
        System.out.println("Wants ice: " + wantsIce);
        System.out.println("Order subtotal: $" + (quantity * price));

        scan.close();
    }
}

ScannerBasics.main(null);

Each call picked up exactly where the previous one stopped. A `Scanner` keeps a **position** in the input and moves forward — it never re-reads what it already consumed.

### The real keyboard version

Outside a notebook — in a `.java` file you run from a terminal or an IDE — this is what you'd write. **Do not paste this into a cell here; it will hang.**

```java
import java.util.Scanner;

public class RealOrderScreen {
    public static void main(String[] args) {
        Scanner scan = new Scanner(System.in);          // <-- the only line that differs

        System.out.print("Customer name: ");
        String name = scan.nextLine();

        System.out.print("How many drinks? ");
        int quantity = scan.nextInt();

        System.out.println("Thanks, " + name + "! " + quantity + " drinks coming up.");
        scan.close();
    }
}
```

Notice the `System.out.print` before each read — no newline, so the cursor waits on the same line as the prompt. That's the `print`/`println` distinction from 1.3 doing real work.

---

## Part 7 — `next()` vs `nextLine()`

These two look interchangeable and are not.

- **`next()`** grabs one **token** — a run of characters ending at the next space. `"Brown Sugar"` gives you just `Brown`.
- **`nextLine()`** grabs everything from the current position to the end of the line.

A drink name like `"Brown Sugar Milk Tea"` has spaces in it, so `next()` would only capture the first word.

In [ ]:
import java.util.Scanner;

public class TokensVsLines {
    public static void main(String[] args) {
        Scanner a = new Scanner("Brown Sugar Milk Tea");
        System.out.println("next():     [" + a.next() + "]");        // just one word
        System.out.println("nextLine(): [" + a.nextLine() + "]");    // whatever is left
        a.close();

        System.out.println();

        Scanner b = new Scanner("Brown Sugar Milk Tea");
        System.out.println("nextLine(): [" + b.nextLine() + "]");    // the whole thing
        b.close();
    }
}

TokensVsLines.main(null);

Look closely at the first `nextLine()` result — there's a **leading space** inside the brackets. `next()` consumed `Brown` and stopped *before* the space, leaving it behind for the next read. Scanners are precise about position, which is exactly what makes the next section bite.

---

## Part 8 — The `nextInt()` then `nextLine()` trap

This is one of the most frequently tested `Scanner` errors, and one of the most common real bugs students hit. Learn it now and you'll recognize it forever.

When a customer types `3` and presses Enter, the input actually contains:

```
3⏎Brown Sugar Milk Tea⏎
```

`nextInt()` reads the `3` and **stops immediately after it** — it does not consume the Enter keypress. That newline is still sitting there.

So the next `nextLine()` says "give me the rest of the current line," looks at what's left, finds nothing before the newline, and hands back an **empty String**.

```
after nextInt():  3⏎Brown Sugar Milk Tea⏎
                   ^ position is here, before the newline
nextLine() returns ""   <- the rest of THIS line, which is nothing
```

**The fix:** call an extra `nextLine()` right after any numeric read, purely to throw the leftover newline away.

In [ ]:
import java.util.Scanner;

public class BufferTrap {
    public static void main(String[] args) {
        String input = "3\nBrown Sugar Milk Tea\n";

        // ---------- BROKEN ----------
        Scanner broken = new Scanner(input);
        int    qty1   = broken.nextInt();
        String drink1 = broken.nextLine();      // gets the leftover newline -> empty!
        System.out.println("BROKEN -> qty = " + qty1 + ", drink = [" + drink1 + "]");
        broken.close();

        // ---------- FIXED ----------
        Scanner fixed = new Scanner(input);
        int    qty2   = fixed.nextInt();
        fixed.nextLine();                        // <-- flush the leftover newline
        String drink2 = fixed.nextLine();        // now reads the real line
        System.out.println("FIXED  -> qty = " + qty2 + ", drink = [" + drink2 + "]");
        fixed.close();
    }
}

BufferTrap.main(null);

The broken version printed `drink = []` — an empty String, not an error. **Nothing crashed.** The program ran to completion and produced a drink with no name.

That makes it a **logic error**, the category from 1.1 that no compiler will ever warn you about.

> **Rule of thumb:** any time `nextInt()` or `nextDouble()` is followed by `nextLine()`, add a flush `nextLine()` between them.

---

## Part 9 — The complete order screen

Everything from this lesson in one program: prompts, input, the flush, an accumulator, and a receipt.

In [ ]:
import java.util.Scanner;

public class OrderScreen {

    public static final double TAX_RATE = 0.0775;

    public static void main(String[] args) {
        // Stand-in for a customer typing at the screen and pressing Enter each time
        String customerTyping = "Riley\n2\nBrown Sugar Milk Tea\n5.75\n";
        Scanner scan = new Scanner(customerTyping);

        System.out.print("Customer name: ");
        String name = scan.nextLine();
        System.out.println(name);

        System.out.print("How many drinks? ");
        int quantity = scan.nextInt();
        scan.nextLine();                       // flush after a numeric read
        System.out.println(quantity);

        System.out.print("Which drink? ");
        String drink = scan.nextLine();
        System.out.println(drink);

        System.out.print("Price each: $");
        double price = scan.nextDouble();
        System.out.println(price);

        // Accumulator building the bill
        double total = 0.0;
        total = total + quantity * price;      // subtotal
        double subtotal = total;
        total = total + subtotal * TAX_RATE;   // add tax

        System.out.println();
        System.out.println("=== BYTE-SIZED BOBA ===");
        System.out.println(name + ": " + quantity + "x " + drink);
        System.out.println("Subtotal: $" + subtotal);
        System.out.println("Total:    $" + total);

        scan.close();
    }
}

OrderScreen.main(null);

Swap `customerTyping` for a different order and re-run. The program never changes — only the input does. That's the whole point of Part 6.

---

# Practice: Running the Order Screen

Four tasks, in order.

---

## Hack 1 — Trace before you run

Fill in the final values **before** running the cell. Double-click to edit.

```java
int a = 5;
int b = 3;
a = a + b;
b = a - b;
a = a - b;
```

| Variable | Your prediction | Actual |
|---|---|---|
| `a` | | |
| `b` | | |

Then, in one sentence, describe what this code segment accomplishes overall.

**Your answer:**

In [ ]:
public class MysterySwap {
    public static void main(String[] args) {
        int a = 5;
        int b = 3;

        a = a + b;
        b = a - b;
        a = a - b;

        System.out.println("a = " + a);
        System.out.println("b = " + b);
    }
}

MysterySwap.main(null);

<details>
<summary><b>Check your answer</b></summary>

| Statement | Right side | `a` | `b` |
|---|---|---|---|
| start | | 5 | 3 |
| `a = a + b;` | `5 + 3` = `8` | **8** | 3 |
| `b = a - b;` | `8 - 3` = `5` | 8 | **5** |
| `a = a - b;` | `8 - 5` = `3` | **3** | 5 |

Final: `a = 3`, `b = 5`.

**What it accomplishes:** it swaps `a` and `b` **without a temp variable**, using arithmetic to stash both values inside `a` temporarily.

Clever — but use the temp-variable version from Part 4 in real code. This trick only works for numbers, it breaks on overflow (Topic 1.5), and the next person to read it has to do algebra to understand it. Being clear beats being clever.
</details>

---

## Hack 2 — The register is dropping items

The cell below has **two** logic errors. It compiles, it runs, and both printed numbers are wrong.

The order is one milk tea ($5.75) plus one matcha ($6.50), so the total should be **$12.25**. And the two cups should end up swapped.

Run it, fix both bugs, then fill in the report.

In [ ]:
public class BrokenRegister {
    public static void main(String[] args) {
        // --- Bug 1: the running total ---
        double total  = 0.0;
        double drink1 = 5.75;
        double drink2 = 6.50;

        total = drink1;
        total = drink2;

        System.out.println("Total should be $12.25, got $" + total);

        // --- Bug 2: swapping the mislabeled cups ---
        String cupA = "Taro";
        String cupB = "Matcha";

        cupA = cupB;
        cupB = cupA;

        System.out.println("cupA should be Matcha, got " + cupA);
        System.out.println("cupB should be Taro, got " + cupB);
    }
}

BrokenRegister.main(null);

**Your bug report** (double-click to edit):

| Bug | What went wrong | Your fix |
|---|---|---|
| 1 | | |
| 2 | | |

<details>
<summary><b>Check your answers</b></summary>

**Bug 1 — the accumulator overwrites instead of accumulating.** `total = drink1;` then `total = drink2;` replaces the value each time, so only the last drink survives. Fix by adding to the existing total:

```java
total = total + drink1;
total = total + drink2;
```

**Bug 2 — the swap has no temp variable.** `cupA = cupB;` destroys `"Taro"` before it's saved anywhere. Fix:

```java
String temp = cupA;
cupA = cupB;
cupB = temp;
```

Both bugs share one root cause: **assignment overwrites.** The old value is not kept unless you deliberately keep it.
</details>

---

## Hack 3 — Build your own order reader

Write a program that reads a full order from a simulated input String and prints a receipt.

The input String is provided. Your job is to read it correctly and use the values.

Requirements:

- Read the name with `nextLine()`
- Read the quantity with `nextInt()` — **remember the flush**
- Read the drink name with `nextLine()` (it contains spaces, so `next()` won't do)
- Read the price with `nextDouble()`
- Read whether they want pearls with `nextBoolean()`
- Use an accumulator to build the total, adding **$0.75** only if they wanted pearls (just add it directly for now — `if` statements are Unit 2)
- Print a labeled receipt
- Call `scan.close()` at the end

In [ ]:
import java.util.Scanner;

public class MyOrderReader {
    public static void main(String[] args) {

        String customerTyping = "Jordan\n3\nMatcha Latte\n6.50\ntrue\n";
        Scanner scan = new Scanner(customerTyping);

        // TODO 1: read the name with nextLine()

        // TODO 2: read the quantity with nextInt()

        // TODO 3: flush the leftover newline

        // TODO 4: read the drink name with nextLine()

        // TODO 5: read the price with nextDouble()

        // TODO 6: read the pearls choice with nextBoolean()

        // TODO 7: build the total with an accumulator, adding 0.75 for pearls

        // TODO 8: print a labeled receipt

        // TODO 9: close the scanner

    }
}

MyOrderReader.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
import java.util.Scanner;

public class MyOrderReader {
    public static void main(String[] args) {

        String customerTyping = "Jordan\n3\nMatcha Latte\n6.50\ntrue\n";
        Scanner scan = new Scanner(customerTyping);

        String  name     = scan.nextLine();
        int     quantity = scan.nextInt();
        scan.nextLine();                        // flush
        String  drink    = scan.nextLine();
        double  price    = scan.nextDouble();
        boolean pearls   = scan.nextBoolean();

        double total = 0.0;
        total = total + quantity * price;
        total = total + 0.75;                   // pearls

        System.out.println("=== BYTE-SIZED BOBA ===");
        System.out.println("Customer: " + name);
        System.out.println(quantity + "x " + drink + " @ $" + price);
        System.out.println("Extra pearls: " + pearls);
        System.out.println("Total: $" + total);

        scan.close();
    }
}

MyOrderReader.main(null);
```

Total: `3 * 6.50 = 19.50`, plus `0.75`, gives `$20.25`. If you got `$0.75`, your accumulator overwrote instead of adding. If your drink name came out empty, you skipped the flush.
</details>

---

## Hack 4 — Diagnose the empty drink

A classmate's order screen prints this:

```
Customer: Sam
Quantity: 4
Drink: []
```

Their code:

```java
Scanner scan = new Scanner("Sam\n4\nThai Tea\n");
String name  = scan.nextLine();
int quantity = scan.nextInt();
String drink = scan.nextLine();
```

Answer these in the markdown below (double-click to edit):

1. Why is the drink name empty?
2. Which error type is this — compile-time, run-time, or logic? Explain how you know.
3. Write the corrected code.

**Your answers:**

1.

2.

3.

<details>
<summary><b>Check your answers</b></summary>

1. `nextInt()` reads the `4` and stops **before** the newline that follows it. The newline is still waiting in the input. When `nextLine()` runs, it returns "the rest of the current line," which is empty, and *then* moves past the newline. The drink name is never reached.

2. **Logic error.** The program compiled and ran to completion with no error message — it just produced a wrong result. A compile-time error would have prevented it from running at all, and a run-time error would have crashed it partway.

3. Add a flush:

```java
Scanner scan = new Scanner("Sam\n4\nThai Tea\n");
String name  = scan.nextLine();
int quantity = scan.nextInt();
scan.nextLine();                 // consume the leftover newline
String drink = scan.nextLine();  // now reads "Thai Tea"
```
</details>

---

# Self-Check: AP-style questions

**1.** What is the value of `x` after this code segment?

```java
int x = 10;
int y = 4;
x = x - y;
y = x + y;
x = y - x;
```

&nbsp;&nbsp;(A) `4` &nbsp;&nbsp; (B) `6` &nbsp;&nbsp; (C) `10` &nbsp;&nbsp; (D) `16`

<details><summary>Answer</summary>

**(A) 4**. Trace it: `x = 10 - 4` gives `6`. Then `y = 6 + 4` gives `10`. Then `x = 10 - 6` gives `4`. The trap is using the original `x` of 10 in the last line instead of the updated `6`.
</details>

---

**2.** Which code segment correctly swaps the values of `p` and `q`?

&nbsp;&nbsp;(A) `p = q; q = p;`
&nbsp;&nbsp;(B) `int t = p; p = q; q = t;`
&nbsp;&nbsp;(C) `int t = p; q = p; p = t;`
&nbsp;&nbsp;(D) `p = q; q = t;`

<details><summary>Answer</summary>

**(B)**. It parks `p` in `t` before overwriting, then restores it into `q`. (A) loses `p` on the first line so both end up as the original `q`. (C) saves `p` but then copies `p` into `q` instead of `q` into `p`. (D) uses `t` without ever declaring or assigning it.
</details>

---

**3.** After `double total = 0.0;`, which statement correctly adds `price` to a running total?

&nbsp;&nbsp;(A) `total = price;`
&nbsp;&nbsp;(B) `price = total + price;`
&nbsp;&nbsp;(C) `total = total + price;`
&nbsp;&nbsp;(D) `total + price;`

<details><summary>Answer</summary>

**(C)**. This is the accumulator pattern. (A) overwrites the total. (B) accumulates into the wrong variable. (D) isn't a valid statement at all — it computes a value and discards it.
</details>

---

**4.** Which assignment causes a compile-time error?

&nbsp;&nbsp;(A) `double d = 5;`
&nbsp;&nbsp;(B) `int i = 5;`
&nbsp;&nbsp;(C) `int i = 5.0;`
&nbsp;&nbsp;(D) `double d = 5.0;`

<details><summary>Answer</summary>

**(C)**. A `double` value can't be stored in an `int` variable, because the fraction would be lost. (A) is fine — an `int` widens into a `double` automatically, storing `5.0`.
</details>

---

**5.** Given `Scanner scan = new Scanner("Taro Milk Tea");`, what does `scan.next()` return?

&nbsp;&nbsp;(A) `Taro Milk Tea` &nbsp;&nbsp; (B) `Taro` &nbsp;&nbsp; (C) `T` &nbsp;&nbsp; (D) An empty String

<details><summary>Answer</summary>

**(B) `Taro`**. `next()` returns a single token and stops at whitespace. To capture the whole line including spaces, use `nextLine()`.
</details>

---

**6.** A program contains:

```java
int count = scan.nextInt();
String label = scan.nextLine();
```

The input is `7` followed by Enter, then `Large Cup`. What does `label` contain?

&nbsp;&nbsp;(A) `Large Cup`
&nbsp;&nbsp;(B) `7`
&nbsp;&nbsp;(C) An empty String
&nbsp;&nbsp;(D) The program throws an exception

<details><summary>Answer</summary>

**(C) an empty String**. `nextInt()` stops right after the `7` without consuming the newline. `nextLine()` then returns the remainder of that line, which is nothing. The fix is an extra `scan.nextLine()` between them to discard the newline.
</details>

---

**7.** Consider `total = total + quantity * price;` where `total` is `10.0`, `quantity` is `2`, and `price` is `3.0`. What is stored in `total`?

&nbsp;&nbsp;(A) `16.0` &nbsp;&nbsp; (B) `36.0` &nbsp;&nbsp; (C) `15.0` &nbsp;&nbsp; (D) `10.0`

<details><summary>Answer</summary>

**(A) 16.0**. The entire right side evaluates first, using precedence from 1.3: `quantity * price` is `6.0`, then `10.0 + 6.0` is `16.0`. Only then is the result stored. Answer (B) comes from adding before multiplying.
</details>

---

# Closing time

### Vocabulary to know cold

| Term | One-line definition |
|---|---|
| Assignment statement | `variable = expression;` — evaluates the right side fully, then stores it |
| Accumulator | A variable that repeatedly adds to itself, as in `total = total + x` |
| Temp variable | A holding spot used to preserve a value before it gets overwritten |
| `Scanner` | A library class that reads values from an input source |
| Token | One chunk of input ending at whitespace — what `next()` returns |
| `nextLine()` | Returns the rest of the current line, spaces included |
| Flush | An extra `nextLine()` call used to discard a leftover newline |

### The five things that will show up on the exam

1. The **entire right side** is evaluated before anything is stored.
2. Assignment **overwrites** — the old value is gone unless you saved it first.
3. Swapping needs a **temp variable**.
4. `total = total + x` accumulates; `total = x` overwrites.
5. `nextInt()` followed by `nextLine()` returns an **empty String** unless you flush.

### Before you submit, check that you:

- [ ] Ran every code cell top to bottom
- [ ] Predicted Hack 1 **before** running it, and explained what the segment accomplishes
- [ ] Fixed both bugs in Hack 2 and got `$12.25` with the cups swapped
- [ ] Completed all nine TODOs in Hack 3 and got `$20.25`
- [ ] Answered all three questions in Hack 4
- [ ] Attempted all seven self-check questions before revealing answers

### Next up

**1.5 — Casting and Range of Variables.** You've now hit the same wall twice: `int rounded = price;` won't compile, and `25 / 2` throws away the half. Both have the same fix, and it's called casting. You'll also find out what happens when a number gets too big for the container you put it in.

See you at the next shift 🧋